# Crypto and Stablecoin Liquidity Pulse

This notebook uses real public crypto and DeFi data. It is descriptive and not investment advice.

Sources:

- CoinGecko market chart for BTC/ETH price history;
- DeFiLlama stablecoin data for liquidity context.

No synthetic fallback is used.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(repo_root)}")
    return path


## 1. Fetch real BTC and ETH market charts


In [ ]:
coins = ["bitcoin", "ethereum"]
frames = [fetch_coingecko_market_chart(c, days=365) for c in coins]
prices = pd.concat(frames, ignore_index=True)
prices.head(20)


## 2. Audit real price table


In [ ]:
audit = source_audit_table(prices, value_col="price", entity_col="coin_id", time_col="date")
audit


## 3. Decompose crypto price series


In [ ]:
components = decompose_table(prices, entity_col="coin_id", time_col="date", value_col="price", method="MA_BASELINE", period=7, trend_window=21, transform="log")
summary = editorial_priority(component_summary(components, entity_col="coin_id", time_col="date"), entity_col="coin_id")
summary


## 4. Residual events


In [ ]:
events = residual_event_table(components, entity_col="coin_id", time_col="date", top_n=20)
events


## 5. Fetch stablecoin context from DeFiLlama

This cell may require schema adjustment if the DeFiLlama stablecoin endpoint changes. It fails explicitly rather than using a fake table.


In [ ]:
try:
    stable_chains = fetch_defillama_stablecoin_chains()
except HotTrendDataError as exc:
    stable_chains = pd.DataFrame([{"status": "stablecoin_fetch_failed", "reason": str(exc)}])
stable_chains.head(20)


## 6. Publication guardrails


In [ ]:
guardrails = article_language_guardrails()
guardrails


In [ ]:
save_table(audit, "06_crypto_price_audit")
save_table(summary, "06_crypto_price_summary")
save_table(events, "06_crypto_price_residual_events")
save_table(stable_chains, "06_defillama_stablecoin_context")
save_table(guardrails, "06_crypto_guardrails")
